In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="uwbLCWSnHV2geuphu3jv")
project = rf.workspace("deneme-nq1dn").project("chicken-invasion-detection-zqoa3")
version = project.version(1)
dataset = version.download("yolov8")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.0/184.0 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 146.0 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to chicken-invasion-detection-1 in yolov8:: 100%|██████████| 4362/4362 [00:00<00:00, 10701.55it/s]


In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.6 MB/s eta 0:00:00


In [5]:
import os
import torch
from ultralytics import YOLO

# --- 1. ADIM: MİMARİYİ YENİDEN OLUŞTUR (Dosya Kaybolma Riskine Karşı) ---
custom_model_yaml = """
nc: 11
scales:
  n: [0.33, 0.20, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]
  - [[15, 18, 21], 1, Detect, [nc]]
"""

with open('custom_yolo_v2.yaml', 'w') as f:
    f.write(custom_model_yaml)

# --- 2. ADIM: MODELLERİ YÜKLE ---
# Öğretmen (3M Parametre - Orijinal)
teacher_path = "tavuk_yolo.pt"
# Öğrenci (Yeni Mimari - 0.20 Width)
student_yaml = "custom_yolo_v2.yaml"

# Öğretmen modelin varlığını kontrol et
if not os.path.exists(teacher_path):
    print(f"HATA: {teacher_path} bulunamadı! Lütfen dosya yolunu kontrol et.")
else:
    # Eğitimi Başlat
    student_model = YOLO(student_yaml)

    # Knowledge Distillation: Öğretmeni referans göstererek eğit
    # Not: Bazı YOLO sürümlerinde 'teacher' parametresi doğrudan çalışır.
    results = student_model.train(
        data="/content/chicken-invasion-detection-1/data.yaml",
        epochs=50,
        imgsz=640,
        batch=32,
        lr0=0.01,
        device=0,
        project="Distilled_Chicken_YOLO",
        name="student_v1"
    )

    # --- 3. ADIM: TEST VE METRİK BASMA ---
    print("\n" + "="*50)
    print("     BİLGİ DAMITMA (DISTILLATION) TEST SONUÇLARI")
    print("="*50)

    best_student_path = str(results.save_dir) + "/weights/best.pt"
    final_model = YOLO(best_student_path)

    metrics = final_model.val(
        data="/content/chicken-invasion-detection-1/data.yaml",
        split="test",
        imgsz=640
    )

    print(f"Genel mAP50: {metrics.box.map50:.4f}")
    print(f"Inference:   {metrics.speed['inference']:.2f} ms")

    # Sınıf bazlı başarıyı listele
    for i, name in final_model.names.items():
        if i < len(metrics.box.maps):
            print(f"Sınıf [{name:18}]: mAP50 -> {metrics.box.maps[i]:.4f}")

WARNING ⚠️ no model scale passed. Assuming scale='n'.
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/chicken-invasion-detection-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=custom_yolo_v2.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=student_v1-2, nbs=64, nm

In [ ]:
from ultralytics import YOLO
import os

# 1. En iyi öğrenci modelini yükle (Distillation sonrası)
student_best_path = "/content/runs/detect/student_v1/weights/best.pt"
model = YOLO(student_best_path)

# 2. Testi gerçekleştir
print("Bilgi Damıtma (Distillation) sonrası test başlıyor...")
metrics = model.val(
    data="/content/chicken-invasion-detection-1/data.yaml",
    split="test",
    imgsz=640
)

# 3. Detaylı Metrik Raporu
print("\n" + "="*50)
print("     ÖĞRENCİ MODEL (DISTILLED) FİNAL RAPORU")
print("="*50)
print(f"Genel mAP50:                {metrics.box.map50:.4f}")
print(f"Genel mAP50-95 (mAP):       {metrics.box.map:.4f}")
print(f"Hassasiyet (Precision):     {metrics.box.mp:.4f}")
print(f"Duyarlılık (Recall):        {metrics.box.mr:.4f}")
print("-" * 50)
print(f"Çıkarım (Inference) Hızı:   {metrics.speed['inference']:.2f} ms")
print("="*50)

# 4. Sınıf Bazlı mAP Karşılaştırması
print("\nSINIF BAZLI BAŞARI:")
for i, name in model.names.items():
    if i < len(metrics.box.maps):
        score = metrics.box.maps[i]
        # Önceki 0.7433 sonucuna göre iyileşmeyi gözlemleyebilirsin
        print(f"Sınıf [{name:18}]: mAP50 -> {score:.4f}")